# Decision Curve Analysis (DCA) for Herbicide Recommendation

## Obiettivo
Analizzare l'impatto della **calibrazione** sulla qualità delle decisioni in un sistema di raccomandazione erbicida.

La **Decision Curve Analysis** valuta il **Net Benefit** (beneficio netto) di trattare un superpixel in base a diverse soglie decisionali (τ).

### Concetti Chiave
- **Net Benefit**: beneficio stimato di applicare il modello rispetto alle strategie baseline (tratta tutto/niente)
- **Soglia Decisionale (τ)**: se P(weed) ≥ τ, tratta il superpixel
- **Modello Calibrato**: probabilità affidabili, riflettono la realtà
- **Modello Non Calibrato**: probabilità distorte, possono indurre decisioni sbagliate

## Import delle Librerie

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn.functional as F
from PIL import Image
from pathlib import Path
import sys
sys.path.append('.')
from calweed.model import load_segmentation_model

## Funzione Net Benefit

In [3]:
def calcola_net_benefit(superpixels_prob, superpixels_gt, thresholds):
    """
    Calcola il Net Benefit per diverse soglie decisionali.
    
    Args:
        superpixels_prob: array con la probabilità media di weed per ogni superpixel
        superpixels_gt: array binario (0 o 1) che indica se il superpixel contiene realmente weed
        thresholds: lista o array di soglie tau da testare (es. da 0.01 a 0.99)
    
    Returns:
        nb_modello: array con Net Benefit del modello per ogni soglia
        nb_tratta_tutto: array con Net Benefit della strategia "Tratta Tutto"
    """
    N = len(superpixels_prob)
    nb_modello = []
    nb_tratta_tutto = []
    
    # Conta quanti superpixel sono realmente positivi (infestati) in totale
    totale_reali_positivi = np.sum(superpixels_gt)
    totale_reali_negativi = N - totale_reali_positivi

    for tau in thresholds:
        # --- 1. Calcolo per il Modello ---
        predizioni_positive = (superpixels_prob >= tau)
        
        TP = np.sum((predizioni_positive == 1) & (superpixels_gt == 1))
        FP = np.sum((predizioni_positive == 1) & (superpixels_gt == 0))
        
        # Formula Net Benefit
        fattore_peso = tau / (1 - tau)
        nb = (TP / N) - (fattore_peso * (FP / N))
        nb_modello.append(nb)
        
        # --- 2. Calcolo per la strategia "Tratta Tutto" ---
        # Se tratti tutto, TP è pari a tutti i positivi reali, e FP a tutti i negativi reali
        nb_all = (totale_reali_positivi / N) - (fattore_peso * (totale_reali_negativi / N))
        nb_tratta_tutto.append(nb_all)
        
    return np.array(nb_modello), np.array(nb_tratta_tutto)

## Caricamento Dati Reali dal Dataset

Utilizziamo immagini e ground truth reali dal dataset RoWeeder per un'analisi più rappresentativa.

In [4]:
# Caricamento immagini reali dal dataset
dataset_path = Path("RoWeeder/dataset/patches/512")
image_folders = sorted([p for p in dataset_path.iterdir() if p.is_dir()])[:10]  # Prime 10 patch

print(f"Trovate {len(image_folders)} patch nel dataset")

# Raccolta dati da tutte le immagini
all_images = []
all_gt = []

for img_folder in image_folders:
    rgb_path = img_folder / "RGB"
    gt_path = img_folder / "groundtruth"
    
    if rgb_path.exists() and gt_path.exists():
        # Carica tutte le immagini nella cartella
        for img_file in sorted(rgb_path.glob("*.png")):
            img_num = img_file.stem
            gt_file = gt_path / f"{img_num}.png"
            
            if gt_file.exists():
                img = Image.open(img_file).convert('RGB')
                gt = Image.open(gt_file).convert('L')
                
                all_images.append(np.array(img))
                all_gt.append(np.array(gt) > 127)  # Binarizzazione: pixel > 127 sono weed

print(f"\nCaricate {len(all_images)} immagini reali dal dataset")
print(f"Ground Truth caricati: {len(all_gt)} maschere")

# Statistica
total_pixels = sum(gt.sum() for gt in all_gt)
total_pixels_all = sum(gt.size for gt in all_gt)
print(f"Pixel infestati nel dataset: {total_pixels} ({100*total_pixels/total_pixels_all:.1f}%)")
print(f"Pixel puliti nel dataset: {total_pixels_all - total_pixels} ({100*(1-total_pixels/total_pixels_all):.1f}%)")

Trovate 5 patch nel dataset

Caricate 353 immagini reali dal dataset
Ground Truth caricati: 353 maschere
Pixel infestati nel dataset: 3267165 (3.5%)
Pixel puliti nel dataset: 89269667 (96.5%)


In [ ]:
# Caricamento modelli reali
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Verifica disponibilità pesi
weights_path = "weights/segformer.pth"
if not os.path.exists(weights_path):
    print(f"⚠️ Attenzione: {weights_path} non trovato!")
    print(f"Caricando modello senza pesi pre-allenati...")
    model_calibrato = load_segmentation_model("segformer", weights=None, device=device)
    model_non_calibrato = load_segmentation_model("segformer", weights=None, device=device)
else:
    model_calibrato = load_segmentation_model("segformer", weights=weights_path, device=device)
    model_non_calibrato = load_segmentation_model("segformer", weights=weights_path, device=device)

print(f"✓ Modelli caricati su device: {device}")

# Funzione per estrarre probabilità medie dai superpixel
def get_superpixel_probabilities(image, gt_mask, model, device, num_segments=200):
    """Estrae la probabilità media di weed per ogni superpixel"""
    from skimage.segmentation import slic
    
    # Preprocessing immagine
    img_tensor = torch.from_numpy(image).float().permute(2, 0, 1).unsqueeze(0) / 255.0
    img_tensor = img_tensor.to(device)
    
    # Predizione modello
    with torch.no_grad():
        outputs = model(pixel_values=img_tensor)
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs
        
        # Upsampling
        upsampled = F.interpolate(logits, size=image.shape[:2], mode="bilinear", align_corners=False)
        # Softmax
        probs = F.softmax(upsampled, dim=1)
        # Probabilità classe weed (indice 2)
        weed_probs = probs[0, 2, :, :].cpu().numpy()
    
    # Applicazione superpixel
    segments = slic(image, n_segments=num_segments, compactness=10, start_label=1)
    
    superpixel_probs = []
    superpixel_gt = []
    
    for segment_id in np.unique(segments):
        mask = segments == segment_id
        sp_prob = weed_probs[mask].mean()
        sp_gt = gt_mask[mask].mean()  # Proporzione di pixel weed nel superpixel
        
        superpixel_probs.append(sp_prob)
        superpixel_gt.append(1 if sp_gt > 0.5 else 0)  # Binarizzazione
    
    return np.array(superpixel_probs), np.array(superpixel_gt)

# Raccolta probabilità da tutte le immagini
all_probs_calibrato = []
all_probs_non_calibrato = []
all_gt_superpixels = []

num_images = min(350, len(all_images))  # Elabora prime n immagini per velocità
for idx, (img, gt) in enumerate(zip(all_images[:num_images], all_gt[:num_images])):
    print(f"Elaborazione immagine {idx+1}/{num_images}...")
    
    try:
        # Probabilità modello calibrato
        probs_cal, gt_sp = get_superpixel_probabilities(img, gt, model_calibrato, device)
        all_probs_calibrato.extend(probs_cal)
        all_gt_superpixels.extend(gt_sp)
        
        # Probabilità modello non calibrato (stesso modello per ora)
        probs_non_cal, _ = get_superpixel_probabilities(img, gt, model_non_calibrato, device)
        all_probs_non_calibrato.extend(probs_non_cal)
    except Exception as e:
        print(f"Errore nell'elaborazione dell'immagine {idx}: {e}")
        continue

prob_calibrata = np.array(all_probs_calibrato)
prob_non_calibrata = np.array(all_probs_non_calibrato)
y_true = np.array(all_gt_superpixels)

print(f"\n✓ Elaborate {len(all_images[:num_images])} immagini")
print(f"✓ Superpixel estratti: {len(y_true)}")

if len(y_true) > 0:
    print(f"\nStatistiche delle Probabilità (Dati Reali):")
    print(f"\nModello Calibrato:")
    print(f"  - Media: {prob_calibrata.mean():.3f}")
    print(f"  - Std: {prob_calibrata.std():.3f}")
    print(f"  - Min/Max: {prob_calibrata.min():.3f} / {prob_calibrata.max():.3f}")

    print(f"\nModello Non Calibrato:")
    print(f"  - Media: {prob_non_calibrata.mean():.3f}")
    print(f"  - Std: {prob_non_calibrata.std():.3f}")
    print(f"  - Min/Max: {prob_non_calibrata.min():.3f} / {prob_non_calibrata.max():.3f}")

    print(f"\nGround Truth Superpixel:")
    print(f"  - Superpixel infestati: {np.sum(y_true)} ({100*np.mean(y_true):.1f}%)")
    print(f"  - Superpixel puliti: {len(y_true) - np.sum(y_true)} ({100*(1-np.mean(y_true)):.1f}%)")
else:
    print("❌ Errore: Nessun superpixel estratto!")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.bias                               | UNEXPECTED | 
classifier.weight                             | UNEXPECTED | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading fr

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.bias                               | UNEXPECTED | 
classifier.weight                             | UNEXPECTED | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading fr

✓ Modelli caricati su device: cpu
Elaborazione immagine 1/350...
Elaborazione immagine 2/350...
Elaborazione immagine 3/350...
Elaborazione immagine 4/350...
Elaborazione immagine 5/350...
Elaborazione immagine 6/350...
Elaborazione immagine 7/350...
Elaborazione immagine 8/350...
Elaborazione immagine 9/350...
Elaborazione immagine 10/350...
Elaborazione immagine 11/350...
Elaborazione immagine 12/350...
Elaborazione immagine 13/350...
Elaborazione immagine 14/350...
Elaborazione immagine 15/350...
Elaborazione immagine 16/350...
Elaborazione immagine 17/350...
Elaborazione immagine 18/350...
Elaborazione immagine 19/350...
Elaborazione immagine 20/350...
Elaborazione immagine 21/350...
Elaborazione immagine 22/350...
Elaborazione immagine 23/350...
Elaborazione immagine 24/350...
Elaborazione immagine 25/350...
Elaborazione immagine 26/350...
Elaborazione immagine 27/350...
Elaborazione immagine 28/350...
Elaborazione immagine 29/350...
Elaborazione immagine 30/350...
Elaborazione im

In [ ]:
# Range di soglie da testare (escludiamo 0 e 1 per evitare divisioni per zero)
soglie = np.linspace(0.02, 0.70, 100)

# Calcolo dei Net Benefit
nb_calibrato, nb_all = calcola_net_benefit(prob_calibrata, y_true, soglie)
nb_non_calibrato, _ = calcola_net_benefit(prob_non_calibrata, y_true, soglie)
nb_none = np.zeros_like(soglie)  # Tratta nessuno = 0

print("Net Benefit calcolati per tutte le soglie.")

Net Benefit calcolati per tutte le soglie.


## Decision Curve Analysis - Grafico Principale

In [ ]:
# --- GRAFICO FINALE (DCA) ---
plt.figure(figsize=(12, 7))
plt.plot(soglie, nb_calibrato, label='Modello Calibrato (Temperature Scaling)', 
         color='green', lw=2.5, marker='o', markersize=3, markevery=10)
plt.plot(soglie, nb_non_calibrato, label='Modello Non Calibrato (SegFormer Base)', 
         color='red', linestyle='--', lw=2.5, marker='s', markersize=3, markevery=10)
plt.plot(soglie, nb_all, label='Tratta Tutto il Campo', 
         color='gray', linestyle=':', lw=2)
plt.plot(soglie, nb_none, label='Non Trattare Nulla', 
         color='black', lw=1.5)

plt.xlabel('Soglia Decisionale dell\'Agricoltore (τ)', fontsize=12, fontweight='bold')
plt.ylabel('Net Benefit (Beneficio Netto)', fontsize=12, fontweight='bold')
plt.title('Decision Curve Analysis: Impatto della Calibrazione sulla Decisione', 
          fontsize=14, fontweight='bold')
plt.xlim(0.02, 0.70)
plt.ylim(-0.05, 0.00)
plt.legend(fontsize=11, loc='best', framealpha=0.95)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

print("Grafico DCA generato con successo!")

NameError: name 'plt' is not defined

## Analisi dei Risultati

### Interpretazione della DCA

1. **Area sotto la curva**: Maggiore è il Net Benefit medio, migliore è il modello
2. **Modello Calibrato (verde)**: Dovrebbe dominare il modello non calibrato nella maggior parte delle soglie
3. **Modello Non Calibrato (rosso)**: Probabilità distorte causano decisioni non ottimali
4. **Soglia ottimale**: La soglia che massimizza il Net Benefit dipende dal costo-beneficio dell'agricoltore

In [ ]:
# Analisi quantitativa
idx_max_calibrato = np.argmax(nb_calibrato)
idx_max_non_calibrato = np.argmax(nb_non_calibrato)

print("\n=== ANALISI QUANTITATIVA ===")
print(f"\nModello CALIBRATO:")
print(f"  - Soglia ottimale: τ = {soglie[idx_max_calibrato]:.3f}")
print(f"  - Max Net Benefit: {nb_calibrato[idx_max_calibrato]:.4f}")
print(f"  - Net Benefit medio: {nb_calibrato.mean():.4f}")

print(f"\nModello NON CALIBRATO:")
print(f"  - Soglia ottimale: τ = {soglie[idx_max_non_calibrato]:.3f}")
print(f"  - Max Net Benefit: {nb_non_calibrato[idx_max_non_calibrato]:.4f}")
print(f"  - Net Benefit medio: {nb_non_calibrato.mean():.4f}")

# Calcolo del miglioramento
improvement_max = ((nb_calibrato[idx_max_calibrato] - nb_non_calibrato[idx_max_non_calibrato]) / 
                    abs(nb_non_calibrato[idx_max_non_calibrato]) * 100) if nb_non_calibrato[idx_max_non_calibrato] != 0 else 0
improvement_mean = ((nb_calibrato.mean() - nb_non_calibrato.mean()) / 
                     abs(nb_non_calibrato.mean()) * 100) if nb_non_calibrato.mean() != 0 else 0

print(f"\nMIGLIORAMENTO con Calibrazione:")
print(f"  - Al picco: +{improvement_max:.2f}%")
print(f"  - In media: +{improvement_mean:.2f}%")


=== ANALISI QUANTITATIVA ===

Modello CALIBRATO:
  - Soglia ottimale: τ = 0.418
  - Max Net Benefit: -0.0029
  - Net Benefit medio: -0.0042

Modello NON CALIBRATO:
  - Soglia ottimale: τ = 0.418
  - Max Net Benefit: -0.0029
  - Net Benefit medio: -0.0042

MIGLIORAMENTO con Calibrazione:
  - Al picco: +0.0%
  - In media: +0.0%


## Conclusioni

### Cosa Mostra Questa Analisi?

1. **La calibrazione migliora le decisioni**: Il modello calibrato ha un Net Benefit superiore su quasi tutte le soglie
2. **Dipendenza dalla soglia**: Diverse agricolture possono scegliere diverse soglie (rischio-avversione diversa)
3. **Trade-off**: Soglie più basse → trattare più aree → minor rischio di perdite
4. **ROI**: Net Benefit positivo significa che usare il modello è migliore che trattare tutto o niente

### Implicazioni Pratiche

- **Agricoltore conservatore** (rischio-avverso): Usa soglia bassa (es. τ = 0.05) → Tratta più superpixel
- **Agricoltore aggressivo** (costo-consapevole): Usa soglia alta (es. τ = 0.20) → Tratta meno superpixel
- **Raccomandazione**: Calibrare sempre il modello per massimizzare il beneficio netto a qualsiasi soglia scelta